# AMR batch docking — Colab worker

The same real AutoDock Vina 1.2.5 worker as the Kaggle notebook, on the **same** Supabase queue
(1,761 medicines × 4 targets = 7,044 jobs): same code (`src/batchdock`), same pinned Vina binary,
same configuration `vina-1.2.5/exh8/modes9/seed42/batchdock-2`, checked by its SHA-256.

It runs in two stages:

1. **Benchmark**: one slot, exactly 5 real queued jobs, claimed under a lease like any worker. A result is
   stored only if it passes the normal checks, so these are real production results, never test data.
   It then reads them back: configuration hash, artifacts in production storage with matching SHA-256,
   no duplicates, and the measured rate.
2. **Production**: only if the benchmark passed and `RUN_PRODUCTION = True`.

**Before running:** Runtime → Change runtime type → **CPU** (Vina 1.2.5 has no GPU code path).
Secrets (key icon, left sidebar): add `SUPABASE_URL`, `SUPABASE_SECRET_KEY`, `DOCKING_DATABASE_URL`
from the laptop's `docking.env`, and turn on *Notebook access* for each. Then Runtime → Run all.
Keep the tab open: Colab ends idle sessions. One account, one session, within Colab's free limits.


In [ ]:
# ---- Settings (operational only; nothing here can change a docking score) ----
REPO_URL = "https://github.com/VED2107/amr-drug-repurposing.git"
REPO_REF = "master"           # branch, tag or commit SHA
MAX_SLOTS = 4                 # calibration tries 1..MAX_SLOTS parallel Vina processes (1 CPU each)
CALIBRATE = True              # False: skip calibration and use FIXED_SLOTS
FIXED_SLOTS = None            # e.g. 3; None = CPUs - 1
JOBS_PER_SLOT = 3             # calibration jobs per slot per phase (all are real production jobs)
BENCHMARK_JOBS = 5            # stage 1: real queued jobs on one slot
RUN_PRODUCTION = True         # stage 2 runs only if the benchmark passed
SESSION_HOURS = 11.0          # stop claiming before Colab's session limit; running jobs finish first
IDLE_EXIT_SECONDS = 900       # leave when the queue has had nothing to claim for this long
STATUS_EVERY_SECONDS = 600    # print campaign progress this often

# The validated configuration. The worker must reproduce this hash exactly.
EXPECTED_CONFIG_HASH = "5965905fa46e4af8a9486af0edc3de11cf50c7f21326053e1f7266715cbcf2b7"
EXPECTED_VERSION_LABEL = "vina-1.2.5/exh8/modes9/seed42/batchdock-2"

# AutoDock Vina 1.2.5, official static Linux build, pinned by checksum (same as Dockerfile.docking).
VINA_URL = "https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.5/vina_1.2.5_linux_x86_64"
VINA_SHA256 = "fa0126a28a9ea9162d1b161dfa92bc76e632416db28ca246278ea4b2dc6860cb"

import os, time
SESSION_START = time.monotonic()
WORK = "/content/amr-work" if os.path.isdir("/content") else os.path.abspath("colab-work")
os.makedirs(WORK, exist_ok=True)
print("working directory:", WORK)

## 1. CPU resources

In [ ]:
import platform, subprocess
from pathlib import Path

def cgroup_quota():
    try:
        q, p = Path("/sys/fs/cgroup/cpu.max").read_text().split()[:2]
        return None if q == "max" else int(q) / int(p)
    except Exception:
        pass
    try:
        q = int(Path("/sys/fs/cgroup/cpu/cpu.cfs_quota_us").read_text())
        p = int(Path("/sys/fs/cgroup/cpu/cpu.cfs_period_us").read_text())
        return q / p if q > 0 else None
    except Exception:
        return None

affinity = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
quota = cgroup_quota()
CPUS = max(1, min(affinity, int(quota)) if quota else affinity)
model = next((l.split(":", 1)[1].strip() for l in Path("/proc/cpuinfo").read_text().splitlines()
              if l.startswith("model name")), "unknown") if Path("/proc/cpuinfo").exists() else platform.processor()
mem = next((int(l.split()[1]) // 1024 for l in Path("/proc/meminfo").read_text().splitlines()
            if l.startswith("MemTotal")), None) if Path("/proc/meminfo").exists() else None
print(f"os.cpu_count={os.cpu_count()}  affinity={affinity}  cgroup quota={quota}  -> usable CPUs={CPUS}")
print(f"cpu model: {model}   memory: {mem} MiB   python {platform.python_version()}")

## 2. Credentials from Colab Secrets
Never written to the notebook, its output or the repository. Only their presence, and whether they name the production project, is printed.

In [ ]:
SECRET_NAMES = ["SUPABASE_URL", "SUPABASE_SECRET_KEY", "DOCKING_DATABASE_URL"]
try:
    from google.colab import userdata
    for name in SECRET_NAMES:
        try:
            os.environ[name] = userdata.get(name)
        except Exception as exc:
            print(f"secret {name}: not available ({type(exc).__name__}); add it and enable notebook access")
except ImportError:
    print("not on Colab: reading the same names from the environment")
missing = [n for n in SECRET_NAMES if not os.environ.get(n)]
for n in SECRET_NAMES:
    print(f"{n}: {'present' if os.environ.get(n) else 'MISSING'}")
assert not missing, f"add these Colab Secrets (key icon, left sidebar) with notebook access: {missing}"

# ---- Production project guard (added after the 2026-10-03 credential incident) ----
# The project ref is the public hostname prefix, not a secret. Every credential must name it.
EXPECTED_PROJECT_REF = "jmkxlfxkvlsawxtmuugr"
import requests
from urllib.parse import urlparse

def _db_ref(url):
    u = urlparse(url)
    user_ref = (u.username or "").partition(".")[2]          # pooler: postgres.<ref>
    host = (u.hostname or "").split(".")
    return user_ref or (host[1] if host[0] == "db" and len(host) > 1 else "")   # direct: db.<ref>.supabase.co

_refs = {
    "SUPABASE_URL": (urlparse(os.environ["SUPABASE_URL"]).hostname or "").split(".")[0],
    "DOCKING_DATABASE_URL": _db_ref(os.environ["DOCKING_DATABASE_URL"]),
}
for _name, _ref in _refs.items():
    print(f"{_name}: {'production project' if _ref == EXPECTED_PROJECT_REF else 'WRONG PROJECT'}")
assert all(r == EXPECTED_PROJECT_REF for r in _refs.values()), \
    "a credential belongs to a different Supabase project; fix the secrets before running"
_key = os.environ["SUPABASE_SECRET_KEY"]
_resp = requests.get(os.environ["SUPABASE_URL"].rstrip("/") + "/storage/v1/bucket",
                     headers={"apikey": _key, "Authorization": "Bearer " + _key}, timeout=30)
_ok = _resp.status_code == 200 and any(b.get("name") == "docking-artifacts" for b in _resp.json())
print("SUPABASE_SECRET_KEY:", "opens production storage" if _ok else f"REJECTED (HTTP {_resp.status_code})")
assert _ok, "the secret key does not open this project's docking-artifacts bucket"
del _key, _resp


## 3. Worker code
The project's own `src/batchdock` package — the same code the laptop's Docker worker runs.

In [ ]:
import sys
REPO = os.environ.get("AMR_REPO_DIR") or f"{WORK}/amr"
if not os.environ.get("AMR_REPO_DIR"):
    if not os.path.isdir(f"{REPO}/.git"):
        subprocess.run(["git", "init", "-q", REPO], check=True)
        subprocess.run(["git", "-C", REPO, "remote", "add", "origin", REPO_URL], check=True)
    subprocess.run(["git", "-C", REPO, "fetch", "-q", "--depth", "1", "origin", REPO_REF], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", "-q", "--force", "FETCH_HEAD"], check=True)
CODE_COMMIT = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code:", REPO, "commit", CODE_COMMIT or "(not a git checkout)")
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

## 4. Python dependencies: psycopg, RDKit, Meeko
Ligands and receptors are **not** re-prepared here: the prepared PDBQT files are downloaded and checked against their SHA-256. RDKit and Meeko are needed because the worker's modules import them.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "psycopg[binary]==3.3.6", "meeko==0.8.0", "rdkit", "numpy", "scipy", "gemmi",
                "python-dotenv", "PyYAML", "requests"],
               check=True)
import importlib
for mod in ["psycopg", "rdkit", "meeko", "numpy", "scipy", "gemmi", "yaml", "requests"]:
    m = importlib.import_module(mod)
    print(f"{mod:<9} {getattr(m, '__version__', '?')}")
from rdkit import Chem
assert Chem.MolFromSmiles("c1ccccc1O") is not None, "RDKit does not work"
from meeko import MoleculePreparation  # noqa: F401  (import check)
print("RDKit and Meeko import and run")

## 5. AutoDock Vina 1.2.5

In [ ]:
import hashlib, urllib.request
VINA = f"{WORK}/bin/vina"
os.makedirs(os.path.dirname(VINA), exist_ok=True)
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()
if not (os.path.exists(VINA) and sha256(VINA) == VINA_SHA256):
    urllib.request.urlretrieve(VINA_URL, VINA)
got = sha256(VINA)
assert got == VINA_SHA256, f"Vina binary checksum mismatch: {got}"
os.chmod(VINA, 0o755)
ver = subprocess.run([VINA, "--version"], capture_output=True, text=True)
print((ver.stdout or ver.stderr).strip(), "| sha256 ok")
assert "1.2.5" in (ver.stdout + ver.stderr), "not AutoDock Vina 1.2.5"

## 6. Connect to the queue and check the configuration

In [ ]:
os.environ.update({
    "DOCKING_VINA_BIN": VINA,
    "DOCKING_ARTIFACT_DIR": f"{WORK}/docking-cache",   # ligands/ and receptors/ are cached here
    "DOCKING_CPU_PER_JOB": "1",
    "DOCKING_WORKER_KIND": "colab",
    "DOCKING_WORKER_LABEL": os.environ.get("DOCKING_WORKER_LABEL")
                            or f"session-{time.strftime('%m%d%H%M')}",
    "DOCKING_SHUTDOWN_GRACE": "900",   # on stop: running Vina jobs get up to 15 min to finish
})
os.environ.pop("DOCKING_EXHAUSTIVENESS", None)   # the validated value (8) is the default; never override it here
for sub in ("ligands", "receptors"):
    os.makedirs(f"{WORK}/docking-cache/{sub}", exist_ok=True)

from src.logging_utils import setup_logging
from src.batchdock import queue as q
from src.batchdock.config import load_settings, parameters_from_env
from src.batchdock.progress import campaign_status, format_status
setup_logging(Path(REPO) / "logs")

settings = load_settings()
params = parameters_from_env()
assert params.config_hash == EXPECTED_CONFIG_HASH, f"configuration differs from the validated one: {params.config_hash}"
assert params.version_label == EXPECTED_VERSION_LABEL, params.version_label
print("configuration", params.version_label, params.config_hash[:16], "| worker", settings.worker_id)

conn = q.wait_for_db(settings.database_url)
run_id = q.current_full_run(conn, params.config_hash)
run_status = conn.execute("select status from docking.runs where run_id = %s", (run_id,)).fetchone() if run_id else None
print("full run:", run_id, run_status[0] if run_status else None)
assert run_id, "no full run exists for this configuration; nothing to claim"
print(format_status(campaign_status(conn, params.config_hash)))

## 7. Cache the receptors (download once, verify SHA-256)
Ligands are fetched the first time a job needs one, checked against the database checksum, and reused from `docking-cache/ligands` for the rest of the session.

In [ ]:
from src.batchdock.artifacts import ArtifactStore
store = ArtifactStore(settings)
rows = conn.execute(
    """select t.target_id, a.object_key, a.sha256 from docking.targets t
         join docking.artifacts a on a.id = t.prepared_artifact_id
        where t.selected and t.preparation_status = 'READY' order by t.target_id""").fetchall()
for target_id, key, digest in rows:
    path = store.fetch(key, digest)
    print(f"{target_id:<10} {key:<45} sha256 {digest[:12]}… ok ({path.stat().st_size:,} bytes)")
store.close()
conn.close()

## 8. GPU (probe only)

Production stays on CPU AutoDock Vina 1.2.5. Vina 1.2.5 has no GPU code path; a GPU engine (Uni-Dock,
Vina-GPU) is a *different engine*, so its scores would belong to a different configuration and could not be
counted toward this campaign under `vina-1.2.5/exh8/modes9/seed42/batchdock-2`. On this project's GTX 1650 the
Uni-Dock test was also slower than CPU Vina. This cell only reports whether a GPU is attached.

In [ ]:
smi = subprocess.run(["bash", "-lc", "nvidia-smi --query-gpu=name,memory.total --format=csv,noheader"],
                     capture_output=True, text=True)
print("GPU:", smi.stdout.strip() or "none attached", "-> production engine: CPU AutoDock Vina 1.2.5")

## 9. Benchmark: 5 real jobs on one slot
Claims the next 5 queued jobs exactly as any worker does (lease and heartbeat). Interrupting returns unfinished
jobs to the queue. Every result goes through the normal checks; nothing is written that a production worker
would not write. Then the results are read back from the database and the artifacts from production storage.

In [ ]:
import dataclasses
from src.batchdock.worker import Worker
from src.batchdock.artifacts import ArtifactStore

bench_settings = dataclasses.replace(settings, concurrency=1, worker_id=settings.worker_id + "-bench")
t0 = time.monotonic()
bench = Worker(bench_settings, params, max_jobs=BENCHMARK_JOBS, idle_exit_seconds=120).run()
elapsed = time.monotonic() - t0
print("benchmark worker:", bench_settings.worker_id, bench)

c = q.wait_for_db(settings.database_url)
jobs = c.execute("""select j.id, j.status, j.config_hash, j.duration_seconds, r.best_affinity,
                           pa.object_key, pa.sha256, la.object_key, la.sha256
                      from docking.jobs j
                      left join docking.results r on r.job_id = j.id
                      left join docking.artifacts pa on pa.id = r.pose_artifact_id
                      left join docking.artifacts la on la.id = r.log_artifact_id
                     where j.worker_id = %s order by j.id""", (bench_settings.worker_id,)).fetchall()
dup = c.execute("select count(*) from (select job_id from docking.results group by 1 having count(*) > 1) x").fetchone()[0]
c.close()

store = ArtifactStore(settings)
problems = []
for jid, status, chash, dur, best, pkey, psha, lkey, lsha in jobs:
    if status != "COMPLETED":
        problems.append(f"job {jid}: {status}")
        continue
    if chash != EXPECTED_CONFIG_HASH:
        problems.append(f"job {jid}: configuration {chash[:16]}")
    for key, digest in ((pkey, psha), (lkey, lsha)):
        try:
            ok = hashlib.sha256(store._download(key)).hexdigest() == digest   # production storage, not the local copy
        except Exception:
            ok = False
        if not ok:
            problems.append(f"job {jid}: artifact {key} missing from storage or checksum differs")
    print(f"job {jid}: {status}  best {best:.3f} kcal/mol  docking {dur:.0f} s")
store.close()

done = sum(1 for j in jobs if j[1] == "COMPLETED")
print(f"\nbenchmark: {done}/{len(jobs)} completed in {elapsed:.0f} s on 1 slot "
      f"-> {done / elapsed * 60:.2f} jobs/min per slot")
print("duplicate results in the campaign:", dup)
for p in problems:
    print("PROBLEM:", p)
BENCHMARK_PASSED = len(jobs) > 0 and not problems and dup == 0
print("BENCHMARK", "PASSED" if BENCHMARK_PASSED else "FAILED")


## 10. Calibrate parallel slots on real jobs
Colab's free CPU runtime usually has 2 CPUs. Each phase runs real queue jobs and stores their results normally.

In [ ]:
assert BENCHMARK_PASSED, "the benchmark did not pass; production is not started"
assert RUN_PRODUCTION, "RUN_PRODUCTION is False; stopping after the benchmark"
from src.batchdock.calibrate import calibrate
if CALIBRATE:
    report = calibrate(settings, params, max_slots=min(MAX_SLOTS, CPUS), jobs_per_slot=JOBS_PER_SLOT)
    for p in report["phases"]:
        print(p)
    SLOTS = report["chosen_slots"]
    print("chosen:", SLOTS, "slots —", report["reason"])
else:
    SLOTS = FIXED_SLOTS or max(1, CPUS - 1)
    print("calibration skipped; slots =", SLOTS)
assert SLOTS * settings.cpu_per_job <= CPUS, "would oversubscribe the CPU"

## 11. Run until the queue is empty or the session ends
Interrupting the kernel (■) is a safe stop: no new jobs are claimed, running jobs finish, anything unfinished goes back to the queue. If Colab disconnects instead, the leases expire within 3 minutes and other workers pick the jobs up.

In [ ]:
import dataclasses, threading
from src.batchdock.worker import Worker

prod = dataclasses.replace(settings, concurrency=SLOTS)
worker = Worker(prod, params, idle_exit_seconds=IDLE_EXIT_SECONDS)
remaining_s = SESSION_HOURS * 3600 - (time.monotonic() - SESSION_START)
stopper = threading.Timer(max(60, remaining_s), worker.request_stop)
stopper.daemon = True
stopper.start()

def report_progress():
    while not worker.stop.wait(STATUS_EVERY_SECONDS):
        try:
            c = q.wait_for_db(settings.database_url)
            print("\n" + format_status(campaign_status(c, params.config_hash)) + "\n", flush=True)
            c.close()
        except Exception as exc:
            print("status read failed:", exc)
threading.Thread(target=report_progress, daemon=True).start()

print(f"worker {prod.worker_id}: {SLOTS} slots, stops claiming in {remaining_s / 3600:.1f} h")
result = worker.run()     # blocks; Ctrl-C / kernel interrupt = graceful stop
stopper.cancel()
print("this session:", result)

## 11. Campaign status after this session

In [ ]:
c = q.wait_for_db(settings.database_url)
run_id = q.current_full_run(c, params.config_hash)
if run_id:
    q.refresh_run_snapshot(c, run_id)
print(format_status(campaign_status(c, params.config_hash)))
c.close()